# Notebook 2: Tube Detection
**Goal:** Detect the center position of each tube lid in every image.

**Strategy (try in order):**
1. **Hough Circle Transform** — fast, zero training, works if lids are circular/elliptical
2. **YOLOv8-nano** — more robust, ~10min to fine-tune on 56 training images

After EDA you will know which approach is viable. Run both and compare.

# Approach A: Hough Circle Transform

In [ ]:
# ── Install deps (Colab) ─────────────────────────────────────────────────────
!pip install ultralytics -q
!pip install opencv-python-headless -q
import os, cv2, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

ann = pd.read_csv(ANN_CSV)
## Approach A: Hough Circle Transform
def detect_hough(img_path, min_r=20, max_r=80, param1=50, param2=20):
    """
    Detect circular tube lids via HoughCircles.
    Returns list of (cx, cy, radius).

    Tune min_r / max_r after EDA: check bbox_w and bbox_h to estimate lid radius.
    param1: Canny high threshold
    param2: accumulator threshold — lower = more circles (more FP), higher = fewer (more FN)
    """
    img  = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 2)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=40,          # min distance between circle centers
        param1=param1,
        param2=param2,
        minRadius=int(min_r),
        maxRadius=int(max_r)
    )
    if circles is None:
        return []
    circles = np.round(circles[0]).astype(int)
    return [(c[0], c[1], c[2]) for c in circles]   # (cx, cy, r)


# ── Quick visual check on a few images ──────────────────────────────────────
# Estimate radius from annotations
approx_r_min = ann[['bbox_w','bbox_h']].min(axis=1).min() / 2 * 0.6
approx_r_max = ann[['bbox_w','bbox_h']].max(axis=1).max() / 2 * 1.2
print(f"Suggested min_r={approx_r_min:.0f}, max_r={approx_r_max:.0f}")

sample = ann['image'].unique()[:4]
fig, axes = plt.subplots(1, len(sample), figsize=(14,5))

for ax, img_name in zip(axes, sample):
    path = os.path.join(IMG_DIR, img_name)
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    detections = detect_hough(path, min_r=approx_r_min, max_r=approx_r_max)

    ax.imshow(img)
    for (cx, cy, r) in detections:
        circle = plt.Circle((cx, cy), r, color='cyan', fill=False, linewidth=2)
        ax.add_patch(circle)
        ax.plot(cx, cy, 'r+', markersize=10)

    # Also plot GT centers
    gt = ann[ann['image'] == img_name]
    ax.scatter(gt['center_x'], gt['center_y'], c='lime', s=40, marker='x', zorder=5)
    ax.set_title(f"{img_name}\nHough: {len(detections)} | GT: {len(gt)}")
    ax.axis('off')

plt.tight_layout()
plt.show()
# ── Run Hough on all images, collect predictions ─────────────────────────────
# You WILL need to tune param1, param2, min_r, max_r based on the visual above.
# This is normal — Hough requires manual calibration.

HOUGH_PARAMS = dict(min_r=approx_r_min, max_r=approx_r_max, param1=50, param2=28)

hough_preds = []   # list of dicts: {image, center_x, center_y}
for img_name in ann['image'].unique():
    path = os.path.join(IMG_DIR, img_name)
    dets = detect_hough(path, **HOUGH_PARAMS)
    for (cx, cy, r) in dets:
        hough_preds.append({'image': img_name, 'center_x': cx, 'center_y': cy})

hough_df = pd.DataFrame(hough_preds)
print(f"Total Hough detections : {len(hough_df)}")
print(f"Total GT annotations   : {len(ann)}")
hough_df.to_csv('/content/hough_predictions.csv', index=False)
print("Saved to /content/hough_predictions.csv")

# Approach B: YOLOv8-nano

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# ── Build YOLO dataset directory ─────────────────────────────────────────────
YOLO_DIR = '/content/yolo_dataset'
for split in ['train', 'val']:
    os.makedirs(f'{YOLO_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_DIR}/labels/{split}', exist_ok=True)

IMG_W, IMG_H = 640, 480

images = ann['image'].unique()
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
val_set = set(val_imgs)

gt_val = ann[ann['image'].isin(val_set)].reset_index(drop=True)
print(f"Val: {len(val_set)} images, {len(gt_val)} GT tubes")
print(f"Train: {len(train_imgs)} images")

def write_yolo_labels(img_list, split):
    for img_name in img_list:
        # Copy image
        src = os.path.join(IMG_DIR, img_name)
        dst = f'{YOLO_DIR}/images/{split}/{img_name}'
        shutil.copy(src, dst)

        # Write label file (class 0 = tube)
        rows = ann[ann['image'] == img_name]
        label_path = f'{YOLO_DIR}/labels/{split}/{os.path.splitext(img_name)[0]}.txt'
        with open(label_path, 'w') as f:
            for _, r in rows.iterrows():
                # YOLO format: class cx_norm cy_norm w_norm h_norm
                cx = r['center_x'] / IMG_W
                cy = r['center_y'] / IMG_H
                w  = r['bbox_w']   / IMG_W
                h  = r['bbox_h']   / IMG_H
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

write_yolo_labels(train_imgs, 'train')
write_yolo_labels(val_imgs,   'val')
print(f"Train: {len(train_imgs)} images | Val: {len(val_imgs)} images")

# Write dataset yaml
yaml_content = f"""
path: {YOLO_DIR}
train: images/train
val:   images/val
nc: 1
names: ['tube']
"""
with open(f'{YOLO_DIR}/dataset.yaml', 'w') as f:
    f.write(yaml_content)
print("YOLO dataset ready.")

In [ ]:
from ultralytics import YOLO

# ── Train YOLOv8-nano (fast, ~5-10 min on Colab T4) ──────────────────────────
model = YOLO('yolov8n.pt')   # downloads pretrained weights

results = model.train(
    data  = f'{YOLO_DIR}/dataset.yaml',
    epochs= 50,
    imgsz = 640,
    batch = 8,
    name  = 'tube_detector',
    patience = 10,   # early stopping
    verbose  = True
)
print("Training complete.")

In [ ]:
# ── Run YOLO inference on all images ─────────────────────────────────────────
best_model = YOLO('runs/detect/tube_detector/weights/best.pt')

yolo_preds = []
for img_name in ann['image'].unique():
    path    = os.path.join(IMG_DIR, img_name)
    result  = best_model(path, verbose=False)[0]
    boxes   = result.boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        conf = float(box.conf[0])
        yolo_preds.append({'image': img_name, 'center_x': cx, 'center_y': cy,
                           'bbox_x': x1, 'bbox_y': y1,
                           'bbox_w': x2-x1, 'bbox_h': y2-y1, 'conf': conf})

yolo_df = pd.DataFrame(yolo_preds)
print(f"Total YOLO detections : {len(yolo_df)}")
yolo_df.to_csv('/content/yolo_predictions.csv', index=False)
print("Saved to /content/yolo_predictions.csv")